In [2]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import zipfile

In [4]:
train = zipfile.ZipFile('train.zip','r')
train.extractall('train')
train.close()

In [3]:
test = zipfile.ZipFile('test.zip','r')
test.extractall('test')
test.close()

In [5]:
print(len(os.listdir('test/test/')))
print(len(os.listdir('train/train/')))

18
18


In [6]:
train_datagen = ImageDataGenerator(rescale = 1/255,
                                   rotation_range=10,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   zoom_range=0.3,
                                   horizontal_flip=False)

train_generator = train_datagen.flow_from_directory('train/train/',
                                                    target_size=(200,200),
                                                    batch_size=32,
                                                    class_mode='categorical')

Found 3780 images belonging to 18 classes.


In [7]:
class_indices = train_generator.class_indices
class_labels = {v: k for k, v in class_indices.items()}
print("Class : ", class_labels)

Class :  {0: 'ba', 1: 'ca', 2: 'da', 3: 'ga', 4: 'ha', 5: 'ja', 6: 'ka', 7: 'la', 8: 'ma', 9: 'na', 10: 'nga', 11: 'nya', 12: 'pa', 13: 'ra', 14: 'sa', 15: 'ta', 16: 'wa', 17: 'ya'}


**VGG16**

* Input size (200 x 200)
* Dense layer (1024)
* Dropout (0.5)
* learning rate Adam (0.001)

In [8]:
pre_trained_model = VGG16(input_shape=(200,200,3),
                         include_top=False,
                         weights='imagenet')

58889256/58889256 [==============================] - 1s 0us/step


In [9]:
for layer in pre_trained_model.layers:
    layer.trainable = False


# Bangun Model nya
x = layers.Flatten()(pre_trained_model.output)
x = layers.Dense(1024, activation= 'relu')(x)
x = layers.Dropout(0.5)(x)
x= layers.Dense(18, activation='softmax')(x)

model = Model(pre_trained_model.input ,x)

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics = ['accuracy'])
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 200, 200, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 200, 200, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 200, 200, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 100, 100, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 100, 100, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 100, 100, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 50, 50, 128)       0     

In [10]:
model.fit(train_generator, epochs=20, verbose=1)

Epoch 1/20
119/119 [==============================] - 51s 352ms/step - loss: 2.1590 - accuracy: 0.4526
Epoch 2/20
119/119 [==============================] - 41s 342ms/step - loss: 0.7983 - accuracy: 0.7521
Epoch 3/20
119/119 [==============================] - 41s 345ms/step - loss: 0.5433 - accuracy: 0.8360
Epoch 4/20
119/119 [==============================] - 42s 351ms/step - loss: 0.4401 - accuracy: 0.8606
Epoch 5/20
119/119 [==============================] - 40s 336ms/step - loss: 0.4040 - accuracy: 0.8751
Epoch 6/20
119/119 [==============================] - 41s 341ms/step - loss: 0.3436 - accuracy: 0.8894
Epoch 7/20
119/119 [==============================] - 41s 346ms/step - loss: 0.2858 - accuracy: 0.9087
Epoch 8/20
119/119 [==============================] - 42s 351ms/step - loss: 0.2802 - accuracy: 0.9103
Epoch 9/20
119/119 [==============================] - 41s 347ms/step - loss: 0.2896 - accuracy: 0.9077
Epoch 10/20
119/119 [==============================] - 41s 341ms/step - l

In [12]:
test_datagen = ImageDataGenerator(rescale=1/255,)


test_generator = test_datagen.flow_from_directory(
    'test/test/',
    target_size=(200,200),
    batch_size=32,
    class_mode = 'categorical',
    shuffle=False
)

Found 1620 images belonging to 18 classes.


In [13]:
import numpy as np
from sklearn.metrics import classification_report
# Prediksi menggunakan model
predIdxs = model.predict(test_generator)

# Konversi prediksi ke label kelas
predIdxs = np.argmax(predIdxs, axis=1)

# Cetak laporan klasifikasi
print(classification_report(test_generator.classes, predIdxs, target_names=test_generator.class_indices.keys()))

51/51 [==============================] - 10s 188ms/step
              precision    recall  f1-score   support

          ba       0.99      1.00      0.99        90
          ca       1.00      1.00      1.00        90
          da       1.00      1.00      1.00        90
          ga       1.00      0.99      0.99        90
          ha       1.00      1.00      1.00        90
          ja       0.99      1.00      0.99        90
          ka       1.00      1.00      1.00        90
          la       1.00      1.00      1.00        90
          ma       0.98      1.00      0.99        90
          na       1.00      1.00      1.00        90
         nga       0.99      1.00      0.99        90
         nya       0.99      0.99      0.99        90
          pa       1.00      0.98      0.99        90
          ra       1.00      1.00      1.00        90
          sa       1.00      1.00      1.00        90
          ta       0.99      0.86      0.92        90
          wa       1.00  

In [14]:
model.save('VGG16.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


**VGG16 (Percobaan ke 2)**

* Input size (200 x 200)
* Dense layer (1024)
* Dropout (0.5)
* learning rate Adam (0.0001)

In [15]:
pre_trained_model = VGG16(input_shape=(200,200,3),
                         include_top=False,
                         weights='imagenet')

for layer in pre_trained_model.layers:
    layer.trainable = False


# Bangun Model nya
x = layers.Flatten()(pre_trained_model.output)
x = layers.Dense(1024, activation= 'relu')(x)
x = layers.Dropout(0.5)(x)
x= layers.Dense(18, activation='softmax')(x)

model2 = Model(pre_trained_model.input ,x)

model2.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics = ['accuracy'])
model2.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 200, 200, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 200, 200, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 200, 200, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 100, 100, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 100, 100, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 100, 100, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 50, 50, 128)       0   

In [16]:
model2.fit(train_generator, epochs=20, verbose=1)

Epoch 1/20
119/119 [==============================] - 49s 395ms/step - loss: 1.8066 - accuracy: 0.4825
Epoch 2/20
119/119 [==============================] - 41s 343ms/step - loss: 0.7894 - accuracy: 0.8032
Epoch 3/20
119/119 [==============================] - 41s 345ms/step - loss: 0.5112 - accuracy: 0.8735
Epoch 4/20
119/119 [==============================] - 41s 347ms/step - loss: 0.4077 - accuracy: 0.8963
Epoch 5/20
119/119 [==============================] - 41s 342ms/step - loss: 0.3126 - accuracy: 0.9228
Epoch 6/20
119/119 [==============================] - 42s 348ms/step - loss: 0.2876 - accuracy: 0.9238
Epoch 7/20
119/119 [==============================] - 41s 342ms/step - loss: 0.2115 - accuracy: 0.9545
Epoch 8/20
119/119 [==============================] - 41s 346ms/step - loss: 0.2002 - accuracy: 0.9497
Epoch 9/20
119/119 [==============================] - 41s 345ms/step - loss: 0.1904 - accuracy: 0.9521
Epoch 10/20
119/119 [==============================] - 41s 343ms/step - l

In [17]:
test_datagen = ImageDataGenerator(rescale=1/255,)


test_generator = test_datagen.flow_from_directory(
    'test/test/',
    target_size=(200,200),
    batch_size=32,
    class_mode = 'categorical',
    shuffle=False
)

Found 1620 images belonging to 18 classes.


In [18]:
import numpy as np
from sklearn.metrics import classification_report
# Prediksi menggunakan model
predIdxs = model2.predict(test_generator)

# Konversi prediksi ke label kelas
predIdxs = np.argmax(predIdxs, axis=1)

# Cetak laporan klasifikasi
print(classification_report(test_generator.classes, predIdxs, target_names=test_generator.class_indices.keys()))

51/51 [==============================] - 6s 107ms/step
              precision    recall  f1-score   support

          ba       1.00      1.00      1.00        90
          ca       1.00      1.00      1.00        90
          da       1.00      1.00      1.00        90
          ga       1.00      1.00      1.00        90
          ha       1.00      1.00      1.00        90
          ja       1.00      1.00      1.00        90
          ka       1.00      1.00      1.00        90
          la       1.00      1.00      1.00        90
          ma       1.00      1.00      1.00        90
          na       0.98      1.00      0.99        90
         nga       1.00      0.98      0.99        90
         nya       1.00      1.00      1.00        90
          pa       1.00      1.00      1.00        90
          ra       1.00      1.00      1.00        90
          sa       1.00      1.00      1.00        90
          ta       0.98      1.00      0.99        90
          wa       1.00   

In [19]:
model2.save('VGG16(2).h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
